In [1]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [2]:
cd /content/gdrive/My Drive/Colab Notebooks/Drivewise

/content/gdrive/My Drive/Colab Notebooks/Drivewise


# **Installing Required Dependencies**

Imports all the external Python libraries needed in the project through the !pip install command. These include PyMuPDF for importing the PDF documents, langchain and langchain-community to manage the RAG process flow, chromadb for the vector database, and sentence-transformers for the embedding models.

In [3]:
!pip install PyMuPDF langchain langchain-community chromadb sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 86.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 106.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 73.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 122.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 63.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 87.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71

# **Importing Required Libraries**

 Includes the import of fitz, which is PyMuPDF used for text extraction, re, which is regular expressions used for keyword matching, and specific classes from LangChain that include Document, HuggingFaceEmbeddings, and Chroma.

In [4]:
import fitz
import re
import os
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

/tmp/ipykernel_612/2959897675.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


# **Defining the Dataset Metadata**

Generates a configuration list known as DATASETS. For each of the 13 Hyundai brochures PDF files, the configuration defines a metadata entry comprising the car brand, the car model, and the document version.

In [5]:
DATASETS = [
    {"file": "alcazar.pdf", "brand": "Hyundai", "model": "ALCAZAR", "version": "2026.1"},
    {"file": "aura.pdf", "brand": "Hyundai", "model": "AURA", "version": "2026.1"},
    {"file": "creta-ev.pdf", "brand": "Hyundai", "model": "CRETA Electric", "version": "2026.1"},
    {"file": "creta-n-line.pdf", "brand": "Hyundai", "model": "CRETA N Line", "version": "2026.1"},
    {"file": "creta.pdf", "brand": "Hyundai", "model": "CRETA", "version": "2026.1"},
    {"file": "exter.pdf", "brand": "Hyundai", "model": "EXTER", "version": "2026.1"},
    {"file": "grand-i10-nios.pdf", "brand": "Hyundai", "model": "Grand i10 NIOS", "version": "2026.1"},
    {"file": "i20-n-line.pdf", "brand": "Hyundai", "model": "i20 N Line", "version": "2026.1"},
    {"file": "i20.pdf", "brand": "Hyundai", "model": "i20", "version": "2026.1"},
    {"file": "ioniq-5.pdf", "brand": "Hyundai", "model": "IONIQ 5", "version": "2026.1"},
    {"file": "venue-n-line.pdf", "brand": "Hyundai", "model": "VENUE N Line", "version": "2026.1"},
    {"file": "venue.pdf", "brand": "Hyundai", "model": "VENUE", "version": "2026.1"},
    {"file": "verna.pdf", "brand": "Hyundai", "model": "VERNA", "version": "2026.1"}
]

# **Defining Brochure Sections and Keywords**

Introduces the concept of the structured chunking approach to the pipeline. Describes the dictionary (BROCHURE_SECTIONS) where the high-level logical sections (such as "engine and performance" and "safety") correspond to lists of regular expressions.

In [6]:
BROCHURE_SECTIONS = {
    "engine and performance": [r"engine", r"performance", r"powertrain", r"transmission", r"turbo", r"gdi", r"power"],
    "mileage and fuel efficiency": [r"mileage", r"fuel efficiency", r"economy", r"kmpl", r"range", r"charge"],
    "safety": [r"safety", r"airbags", r"adas", r"smartsense", r"braking", r"esc", r"isofix"],
    "dimensions": [r"dimensions", r"wheelbase", r"length", r"width", r"height", r"capacity", r"boot space"],
    "interior and comfort": [r"interior", r"comfort", r"seating", r"cabin", r"upholstery", r"space"],
    "infotainment and connectivity": [r"infotainment", r"connectivity", r"bluelink", r"touchscreen", r"audio", r"bose"]
}

# **Creating the Section Classification Function**

This method accepts an excerpt of text that was extracted from the document and searches through the text for any of the keywords listed in the previous cell. The text is classified based on the number of matching keywords.

In [7]:
def determine_section(text_chunk):
    text_lower = text_chunk.lower()
    best_match = "general specifications"
    max_hits = 0

    for section, keywords in BROCHURE_SECTIONS.items():
        hits = sum(1 for kw in keywords if re.search(r'\b' + kw + r'\b', text_lower))
        if hits > max_hits:
            max_hits = hits
            best_match = section

    return best_match

# **Building the Data Ingestion and Chunking Pipeline**

The process_brochure_directory function loops through your PDF data sets, retrieves the text in chunks via PyMuPDF, strips the short noisy chunks, and passes the rest through the section classifier. The strict metadata (brand, model, section, page number) is tagged to each chunk, and they are turned into vectors by using the HuggingFace BAAI/bge-large-en-v1.5 model.

In [8]:
def process_brochure_directory(dataset_config, persist_directory="./drivewise_vectordb"):
    all_documents = []

    print("Starting Data Ingestion Phase...")

    for car in dataset_config:
        file_path = car["file"]
        if not os.path.exists(file_path):
            print(f"Warning: {file_path} not found. Skipping.")
            continue

        print(f"Processing: {car['brand']} {car['model']}...")
        doc = fitz.open(file_path)

        for page_num in range(len(doc)):
            page = doc[page_num]
            blocks = page.get_text("blocks")

            for block in blocks:
                text = block[4].strip()
                if len(text) < 60:
                    continue

                section = determine_section(text)

                metadata = {
                    "car_brand": car["brand"],
                    "car_model": car["model"],
                    "brochure_section": section,
                    "page_number": page_num + 1,
                    "document_version": car["version"],
                    "source": car["file"]
                }

                langchain_doc = Document(page_content=text, metadata=metadata)
                all_documents.append(langchain_doc)

    print(f"Successfully chunked {len(all_documents)} documents across {len(dataset_config)} brochures.")

    print("Initializing Embedding Model (HuggingFace BGE)...")
    embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-large-en-v1.5")

    print("Building Vector Database...")
    vectordb = Chroma.from_documents(
        documents=all_documents,
        embedding=embeddings,
        persist_directory=persist_directory
    )

    vectordb.persist()
    print(f"Vector Database successfully saved to {persist_directory}")
    return vectordb

# **Executing the Data Ingestion Pipeline**

The system processes all 13 brochures, resulting in the production of 736 text segments, loading the required weights for the embedding model and finally saving the Vector Database at ./drivewise_vectordb.

In [9]:
if __name__ == "__main__":
    vector_store = process_brochure_directory(DATASETS)

Starting Data Ingestion Phase...
Processing: Hyundai ALCAZAR...
Processing: Hyundai AURA...
Processing: Hyundai CRETA Electric...
Processing: Hyundai CRETA N Line...
Processing: Hyundai CRETA...
Processing: Hyundai EXTER...
Processing: Hyundai Grand i10 NIOS...
Processing: Hyundai i20 N Line...
Processing: Hyundai i20...
Processing: Hyundai IONIQ 5...
Processing: Hyundai VENUE N Line...
Processing: Hyundai VENUE...
Processing: Hyundai VERNA...
Successfully chunked 736 documents across 13 brochures.
Initializing Embedding Model (HuggingFace BGE)...


/tmp/ipykernel_612/3242904992.py:41: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-large-en-v1.5")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Building Vector Database...
Vector Database successfully saved to ./drivewise_vectordb


/tmp/ipykernel_612/3242904992.py:50: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectordb.persist()


# **Initializing the Retrieval System**

Responsible for reloading the Chroma database and the HuggingFace embedding model that we created earlier so that our system can be able to receive input from users.

In [10]:
DB_DIRECTORY = "./drivewise_vectordb"

def load_retrieval_system():
    print("Loading Embedding Model (BAAI/bge-large-en-v1.5)...")
    embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-large-en-v1.5")

    if not os.path.exists(DB_DIRECTORY):
        print(f"Error: Vector database not found at {DB_DIRECTORY}.")
        print("Please run the Data Ingestion script first.")
        return None

    print("Loading Vector Database...")
    vector_store = Chroma(persist_directory=DB_DIRECTORY, embedding_function=embeddings)
    print("DriveWise Retrieval System Ready!\n")
    return vector_store

# **Creating the Interactive Query Interface**

Defines the CLI loop (interactive_retrieval_loop), which models the user interface. The script prompts the user for the car brand, model, and the question. Importantly, the script performs strict metadata filtering (matches the exact brand and model) before conducting the similarity search. Lastly, the output is formatted to show the best matching chunks along with the file name, section number, and page number.

In [11]:
def interactive_retrieval_loop(vector_store):
    print("="*60)
    print("   Welcome to the DriveWise: Metadata-Aware Automotive RAG Assistant")
    print("="*60)

    while True:
        print("\n New Query Session")
        brand = input("Enter Car Brand or 'exit' to quit: ").strip()
        if brand.lower() == 'exit':
            break

        model = input("Enter Car Model: ").strip()
        if model.lower() == 'exit':
            break

        query = input(f"What would you like to know about the {brand} {model}?: ").strip()
        if query.lower() == 'exit':
            break

        print("\n[System] Searching brochure database...")

        filter_dict = {
            "$and": [
                {"car_brand": {"$eq": brand}},
                {"car_model": {"$eq": model}}
            ]
        }

        try:
            results = vector_store.similarity_search(
                query=query,
                k=3,
                filter=filter_dict
            )

            if not results:
                print(f"\n[!] No relevant sections found for the {brand} {model}.")
                continue

            print("\n")
            print(f" TOP {len(results)} RETRIEVED BROCHURE CHUNKS")

            for i, doc in enumerate(results, 1):
                meta = doc.metadata
                print(f"\n Chunk {i} ")
                print(f"Source File: {meta.get('source', 'Unknown')}")
                print(f"Section:     {meta.get('brochure_section', 'Unknown').upper()}")
                print(f"Page Number: {meta.get('page_number', 'Unknown')}")
                print(f"Content Excerpt:\n{doc.page_content[:400]}...\n")

        except Exception as e:
            print(f"\n[!] An error occurred during retrieval: {e}")

# **Running the DriveWise: Metadata-Aware Automotive RAG Assistant**

In [12]:
if __name__ == "__main__":
    v_store = load_retrieval_system()
    if v_store:
        interactive_retrieval_loop(v_store)

Loading Embedding Model (BAAI/bge-large-en-v1.5)...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

/tmp/ipykernel_612/1569982184.py:13: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_store = Chroma(persist_directory=DB_DIRECTORY, embedding_function=embeddings)


Loading Vector Database...
DriveWise Retrieval System Ready!

   Welcome to the DriveWise: Metadata-Aware Automotive RAG Assistant

 New Query Session
Enter Car Brand or 'exit' to quit: Hyundai
Enter Car Model: CRETA
What would you like to know about the Hyundai CRETA?: dimensions

[System] Searching brochure database...


 TOP 3 RETRIEVED BROCHURE CHUNKS

 Chunk 1 
Source File: creta.pdf
Section:     DIMENSIONS
Page Number: 17
Content Excerpt:
Dimensions 
 
 
Overall length (mm) 
 
4 330 
 
Overall width (mm) 
 
1 790 
 
Overall height (mm) 
 
1 635^^^ 
 
Wheelbase (mm) 
 
2 610 
 
Fuel tank capacity (l) 
 
50 
 
Engine 
 
 
Configuration 
4 cylinders, 16 valves 
4 cylinders, 16 valves 
4 cylinders, 16 valves
Cam type 
DOHC 
DOHC 
DOHC
Displacement (cm3) 
1 497 
1 493 
1 482...


 Chunk 2 
Source File: creta.pdf
Section:     DIMENSIONS
Page Number: 17
Content Excerpt:
Dimensions 
 
 
Overall length (mm) 
 
4 330 
 
Overall width (mm) 
 
1 790 
 
Overall height (mm) 
 
1 635^^^ 
 
Whee